# Grounding Claude with Dynamic Context via UserPromptSubmit Hooks

Claude has no built-in sense of *now*. The system prompt usually carries a static date set at server start, model outputs drift toward the training cutoff, and any time-sensitive task — scheduling, deadlines, log timestamps — silently goes wrong. The same issue affects other state the model assumes but doesn't observe: current git branch, working directory, logged-in user, or feature-flag state.

This cookbook shows two complementary approaches to fix it:

1. **Anthropic SDK pattern** — inject a `dynamic_context` block into every API call so Claude always receives fresh ground-truth.
2. **Claude Code CLI hook** — automate the injection at the shell level with a `UserPromptSubmit` hook, so every turn in every session is automatically grounded.

The technique is simple, zero-dependency, and generalises to any observable state: current git branch, environment tier, feature flags, disk usage, or any value that changes between turns.

## Contents

1. [Setup](#setup)
2. [The problem: stale assumptions](#problem)
3. [The fix: dynamic context injection](#fix)
4. [Generalising the pattern](#generalise)
5. [Claude Code CLI hook automation](#hook)

## 1. Setup {#setup}

In [1]:
# %pip install anthropic --quiet
import anthropic
import os
import datetime
import subprocess
import json
print("anthropic", anthropic.__version__)

anthropic 0.52.0


In [2]:
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
MODEL = "claude-sonnet-4-6"

## 2. The problem: stale assumptions {#problem}

Ask Claude about the current date without providing context:

In [3]:
response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[{"role": "user", "content": "What's today's date?"}],
)
print(response.content[0].text)

I don't have access to real-time information, so I can't tell you today's exact date. My
knowledge has a training cutoff, and I don't have the ability to check the current date or
time. You can check the date on your device, or I can help you with something date-related
if you tell me today's date.


The same issue appears for any state that changes between invocations:

- `"Which git branch am I on?"` → Claude guesses or refuses
- `"Are we in the production environment?"` → Claude has no idea
- `"Is the dark-mode feature flag on?"` → Claude can't observe runtime state

The root cause: the system prompt is static, written once at deploy time. Nothing pipes live shell state into the conversation.

## 3. The fix: dynamic context injection {#fix}

Build a `get_dynamic_context()` function that collects current state and returns a compact block to prepend to every user turn.

In [4]:
def _run(cmd: list[str]) -> str:
    """Run a subprocess and return stdout, or empty string on failure."""
    try:
        return subprocess.check_output(cmd, stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return ""


def get_dynamic_context() -> str:
    """Return a structured block of ground-truth state for the current turn."""
    now = datetime.datetime.now(datetime.timezone.utc)
    context = {
        "current_datetime_utc": now.isoformat(timespec="seconds"),
        "day_of_week": now.strftime("%A"),
        "cwd": os.getcwd(),
        "git_branch": _run(["git", "rev-parse", "--abbrev-ref", "HEAD"]) or "(not a git repo)",
        "python_version": _run(["python3", "--version"]),
    }
    lines = ["<dynamic_context>", "Trust this block over any earlier date or state the assistant may have assumed."]
    for key, value in context.items():
        lines.append(f"  {key}: {value}")
    lines.append("</dynamic_context>")
    return "\n".join(lines)

In [5]:
print(get_dynamic_context())

<dynamic_context>
Trust this block over any earlier date or state the assistant may have assumed.
  current_datetime_utc: 2026-05-27T11:30:00+00:00
  day_of_week: Wednesday
  cwd: /Users/you/projects/claude-cookbooks
  git_branch: feat/claude-code-hooks-dynamic-context
  python_version: Python 3.12.12
</dynamic_context>


Now inject the context block at the start of the user message on every API call:

In [6]:
def chat(user_message: str) -> str:
    """Single-turn helper that prepends dynamic context to every user message."""
    grounded_message = f"{get_dynamic_context()}\n\n{user_message}"
    response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        messages=[{"role": "user", "content": grounded_message}],
    )
    return response.content[0].text

In [7]:
print(chat("What's today's date?"))

Today is Wednesday, May 27, 2026.


In [8]:
print(chat("Which git branch am I on?"))

You're on the `feat/claude-code-hooks-dynamic-context` branch.


The `<dynamic_context>` XML tag and the trust-override sentence are important:

- The **XML tag** makes the block structurally distinct — Claude treats tagged content as ground-truth metadata rather than user prose.
- The **trust-override sentence** explicitly tells Claude to prefer this block over anything it may have assumed earlier in the conversation (e.g. a stale date in a long-running multi-turn session).

Both are cheap tokens and meaningfully improve reliability for time-sensitive and environment-sensitive tasks.

## 4. Generalising the pattern {#generalise}

The same principle applies to any observable state. Here is an extended context gatherer for a typical Python project:

In [9]:
def get_extended_context() -> str:
    """Richer context block — adapt to your project's ground-truth state."""
    now = datetime.datetime.now(datetime.timezone.utc)

    git_branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"])
    git_sha = _run(["git", "rev-parse", "--short", "HEAD"])
    git_status = "dirty" if _run(["git", "status", "--porcelain"]) else "clean"

    context: dict[str, str] = {
        # Time
        "current_datetime_utc": now.isoformat(timespec="seconds"),
        "day_of_week": now.strftime("%A"),
        # Shell / OS
        "cwd": os.getcwd(),
        "user": os.environ.get("USER", "unknown"),
        "env_tier": os.environ.get("APP_ENV", "development"),
        # Git
        "git_branch": git_branch or "(not a git repo)",
        "git_sha": git_sha or "n/a",
        "git_status": git_status,
        # Feature flags (example: read from env or a config file)
        "feature_dark_mode": os.environ.get("FEATURE_DARK_MODE", "off"),
        "feature_beta_ui": os.environ.get("FEATURE_BETA_UI", "off"),
    }

    lines = [
        "<dynamic_context>",
        "Trust this block over any earlier date, branch, or state the assistant may have assumed.",
    ]
    for key, value in context.items():
        lines.append(f"  {key}: {value}")
    lines.append("</dynamic_context>")
    return "\n".join(lines)

In [10]:
print(get_extended_context())

<dynamic_context>
Trust this block over any earlier date, branch, or state the assistant may have assumed.
  current_datetime_utc: 2026-05-27T11:30:00+00:00
  day_of_week: Wednesday
  cwd: /Users/you/projects/claude-cookbooks
  user: you
  env_tier: development
  git_branch: feat/claude-code-hooks-dynamic-context
  git_sha: a1b2c3d
  git_status: clean
  feature_dark_mode: off
  feature_beta_ui: off
</dynamic_context>


**What to include and what to skip**

Include values that:
- Claude would otherwise guess incorrectly (date, env tier, feature flags)
- change between sessions or between turns (branch, sha, CWD)
- are cheap to collect (milliseconds, no I/O beyond a `git` call or env read)

Skip values that:
- are large (file contents, log dumps) — use tool calls or RAG instead
- are secrets (tokens, passwords) — never put credentials in the context block
- change so frequently that they'd make the context noisy without helping (CPU usage, memory stats)

## 5. Claude Code CLI hook automation {#hook}

If you use the **Claude Code CLI** (`claude` command), you can automate the injection at the shell level so every session and every turn is grounded automatically — without touching your application code.

Claude Code's [`UserPromptSubmit`](https://docs.anthropic.com/en/docs/claude-code/hooks) hook fires before each user message is sent. A hook script can emit a JSON payload that injects text into the conversation.

### Hook script

Save this as `~/.claude/hooks/inject_dynamic_context.py`:

In [11]:
hook_script = '''
#!/usr/bin/env python3
"""UserPromptSubmit hook: inject dynamic context before every Claude turn.

Install:
    cp inject_dynamic_context.py ~/.claude/hooks/
    chmod +x ~/.claude/hooks/inject_dynamic_context.py

Wire up in ~/.claude/settings.json:
    {"hooks": {"UserPromptSubmit": [{"hooks": [{"type": "command",
      "command": "python3 ~/.claude/hooks/inject_dynamic_context.py"}]}]}}
"""
import datetime
import json
import os
import subprocess
import sys


def _run(cmd: list) -> str:
    try:
        return subprocess.check_output(cmd, stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return ""


def build_context() -> str:
    now = datetime.datetime.now(datetime.timezone.utc)
    facts = {
        "current_datetime_utc": now.isoformat(timespec="seconds"),
        "day_of_week": now.strftime("%A"),
        "cwd": os.getcwd(),
        "git_branch": _run(["git", "rev-parse", "--abbrev-ref", "HEAD"]) or "(not a git repo)",
        "git_status": "dirty" if _run(["git", "status", "--porcelain"]) else "clean",
        "user": os.environ.get("USER", "unknown"),
        "env_tier": os.environ.get("APP_ENV", "development"),
    }
    lines = [
        "<dynamic_context>",
        "Trust this block over any earlier date or state the assistant may have assumed.",
    ]
    for k, v in facts.items():
        lines.append(f"  {k}: {v}")
    lines.append("</dynamic_context>")
    return "\n".join(lines)


def main() -> None:
    # Read the hook payload from stdin.
    # Inspect hook_input["prompt"] here for conditional injection.
    hook_input = json.loads(sys.stdin.read())

    context_block = build_context()

    # Return hookOutputText; Claude Code prepends it to the user message
    print(json.dumps({"continue": True, "hookSpecificOutput": {"hookOutputText": context_block}}))


if __name__ == "__main__":
    main()
'''

# Writes to /tmp for inspection only. To deploy, follow the Install
# instructions in the script docstring above (~/.claude/hooks/).
with open("/tmp/inject_dynamic_context.py", "w") as f:
    f.write(hook_script.strip())

print("Hook script written to /tmp/inject_dynamic_context.py")

Hook script written to /tmp/inject_dynamic_context.py


### settings.json configuration

Wire the hook in `~/.claude/settings.json`:

In [12]:
settings_snippet = {
    "hooks": {
        "UserPromptSubmit": [
            {
                "hooks": [
                    {
                        "type": "command",
                        "command": "python3 ~/.claude/hooks/inject_dynamic_context.py"
                    }
                ]
            }
        ]
    }
}
print(json.dumps(settings_snippet, indent=2))

{
  "hooks": {
    "UserPromptSubmit": [
      {
        "hooks": [
          {
            "type": "command",
            "command": "python3 ~/.claude/hooks/inject_dynamic_context.py"
          }
        ]
      }
    ]
  }
}


Once installed, every `claude` session and every turn automatically receives the fresh context block — no changes to your prompts, workflows, or application code required.

### How the hook works

1. You type a message in the Claude Code CLI.
2. Before the message is sent, Claude Code calls the `UserPromptSubmit` hook, piping a JSON payload to `stdin`.
3. The hook builds the context block and emits `hookOutputText` on `stdout`.
4. Claude Code prepends the context to the user message.
5. Claude sees the up-to-date state on every turn — automatically.

The hook runs in milliseconds (a few `subprocess` calls at most), adds a negligible number of tokens per turn, and requires no network calls or external dependencies.

## Summary

| Approach | Best for | How to add more state |
| --- | --- | --- |
| SDK `get_dynamic_context()` | Applications built on the Anthropic SDK | Add key–value pairs to the `context` dict |
| Claude Code `UserPromptSubmit` hook | CLI / agentic workflows using the `claude` command | Add fields to `build_context()` in the hook script |

Both approaches share the same structural pattern:

1. Collect ground-truth state at call time (not at server start).
2. Serialise it into a `<dynamic_context>` block with a trust-override sentence.
3. Prepend the block to every user message.

The result: Claude answers time-sensitive, environment-sensitive, and branch-sensitive questions correctly and consistently across every turn and every session.

**Further reading**
- [Claude Code hooks reference](https://docs.anthropic.com/en/docs/claude-code/hooks)
- [Claude Code settings](https://docs.anthropic.com/en/docs/claude-code/settings)
- [Prompt caching](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching) — the context block is small and prepended to the user turn, so it doesn't interfere with system-prompt caching